In [ ]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import joblib

In [ ]:
data = {

    "company_id": [
        1, 2, 3, 4, 5
    ],


    "company": [
        "Tech Company A",
        "Software Company B",
        "AI Company C",
        "Security Company D",
        "Startup E"
    ],


    "opportunity_title": [
        "AI Internship",
        "Web Developer Training",
        "Data Science Internship",
        "Cyber Security Program",
        "Backend Developer Internship"
    ],


    "required_skills": [

        "python machine learning deep learning tensorflow",

        "html css javascript react frontend",

        "python sql data analysis statistics",

        "linux cybersecurity network security",

        "java python api database backend"
    ]

}

df = pd.DataFrame(data)
df.head()

In [ ]:
vectorizer = TfidfVectorizer()
skill_vectors = vectorizer.fit_transform(
    df["required_skills"]
)
print("Opportunity Gap AI Model trained successfully")

In [ ]:
def analyze_opportunity_gap(
    company_id
):

    opportunity = df[
        df["company_id"] == company_id
    ].iloc[0]


    required_skills = (
        opportunity["required_skills"]
        .split()
    )

    market_skills = [
        "python",
        "javascript",
        "sql",
        "java",
        "html",
        "css"
    ]


    missing_skills = list(
        set(required_skills)
        -
        set(market_skills)
    )


    training_programs = [

        f"{skill} training program"

        for skill in missing_skills
    ]


    return {

        "company_id":
            company_id,


        "opportunity":
            opportunity["opportunity_title"],


        "skills_in_demand":
            required_skills,


        "missing_skills":
            missing_skills,


        "recommended_training_programs":
            training_programs
    }


joblib.dump(
    vectorizer,
    "opportunity_vectorizer.pkl"
)


joblib.dump(
    skill_vectors,
    "opportunity_vectors.pkl"
)


joblib.dump(
    df,
    "opportunity_data.pkl"
)



print("Opportunity Gap AI Model saved successfully")

In [ ]:
result = analyze_opportunity_gap(
    company_id=3
)
result

In [ ]:
from fastapi import FastAPI
import joblib

app = FastAPI(
    title="Opportunity Gap Analysis AI API",
    description="Analyze company skill gaps and recommend training programs",
    version="1.0"
)

vectorizer = joblib.load(
    "opportunity_vectorizer.pkl"
)

skill_vectors = joblib.load(
    "opportunity_vectors.pkl"
)

opportunities = joblib.load(
    "opportunity_data.pkl"
)


print("Opportunity Gap AI Model loaded successfully")

In [ ]:
@app.get("/")
def home():

    return {
        "message": "Hello World"
    }



@app.post("/opportunity-gap")
def opportunity_gap(data: dict):

    company_id = data.get(
        "companyId"
    )


    opportunity = opportunities[
        opportunities["company_id"] == company_id
    ].iloc[0]



    required_skills = (
        opportunity["required_skills"]
        .split()
    )



    # مهارات السوق المتوفرة

    market_skills = data.get(
        "available_skills",
        [
            "python",
            "javascript",
            "sql",
            "java",
            "html",
            "css"
        ]
    )



    missing_skills = list(
        set(required_skills)
        -
        set(
            [
                skill.lower()
                for skill in market_skills
            ]
        )
    )



    training_programs = [

        f"{skill} training program"

        for skill in missing_skills

    ]



    return {


        "companyId":
            company_id,


        "opportunity":
            opportunity["opportunity_title"],


        "suggested_skills_in_demand":
            required_skills,


        "missing_skills":
            missing_skills,


        "recommended_training_programs":
            training_programs
    }

In [ ]:
import nest_asyncio
import uvicorn

nest_asyncio.apply()
config = uvicorn.Config(
    app,
    host="127.0.0.1",
    port=8001,
    log_level="info"
)

server = uvicorn.Server(config)

await server.serve()